# Dog Replacement Comparison Visualization

This notebook creates presentation-friendly bar charts from the completed cycle-consistency evaluation.

It does **not** rerun any model. It only reads `data/outputs/comparison_eval/cycle_metrics.csv` and saves figures under `data/outputs/comparison_eval/figures/`.

The goal is to make the comparison easy to explain: which method wins, what it costs, and which metric drove the result.

## Metrics Used

This notebook uses **cycle-consistency evaluation**. For each method, we run replacement forward and then replace back. The reconstructed image is compared with the original image. A better method should reconstruct the original image more faithfully.

The raw metrics have different units and directions, so the charts convert them into a normalized score inside each comparison group and pair:

- Best method for a metric gets `1.0`.
- Worst method for a metric gets `0.0`.
- Intermediate methods are linearly scaled between `0` and `1`.
- The final `cycle score` is the mean of the three normalized metric scores below.

Metric definitions:

- `boundary_ring_mae`: Mean absolute RGB pixel error around the dog boundary. Lower is better. This is sensitive to halos, white edges, dark seams, and bad blending around fur.
- `edited_region_lpips`: LPIPS perceptual distance inside the edited region. Lower is better. This is closer to human visual perception than raw pixel difference.
- `background_mae`: Mean absolute RGB pixel error in the nearby background region. Lower is better. This catches cases where diffusion or blending damages the wall, grass, floor, or other non-dog content.

Runtime is reported separately as `elapsed_min`. Runtime is **not** included in the cycle score because quality and speed should be interpreted as a tradeoff, not mixed into a single hidden number.


In [ ]:
from pathlib import Path
import csv
import math
import textwrap

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
METRIC_CSV = PROJECT_ROOT / "data" / "outputs" / "comparison_eval" / "cycle_metrics.csv"
FIGURE_DIR = PROJECT_ROOT / "data" / "outputs" / "comparison_eval" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not METRIC_CSV.exists():
    raise FileNotFoundError(f"Missing metrics file: {METRIC_CSV}")

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

NUMERIC_COLUMNS = [
    "boundary_ring_mae",
    "edited_region_lpips",
    "edited_region_ssim",
    "background_mae",
    "dog_region_ssim",
    "elapsed_min",
]

def to_float(value):
    try:
        out = float(value)
    except (TypeError, ValueError):
        return float("nan")
    return out if math.isfinite(out) else float("nan")

rows = []
with open(METRIC_CSV, "r", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        for col in NUMERIC_COLUMNS:
            row[col] = to_float(row.get(col))
        rows.append(row)

PAIR_ORDER = ["pair_1", "pair_2", "face_pair"]

GROUPS = {
    "A": {
        "title": "A. Segmentation Strategy",
        "short": "Segmentation",
        "variants": ["seg_yolo_only", "seg_sam_bbox_prior", "seg_yolo_sam_consensus", "seg_current_conservative"],
        "note": "Which mask-generation strategy gives the cleanest end-to-end replacement?",
    },
    "B": {
        "title": "B. Pose / Geometry Adjustment",
        "short": "Pose",
        "variants": ["pose_face_only", "pose_micro_piecewise", "pose_face_draggan_current"],
        "note": "Among active geometry strategies, compare face only, body pose only, and face + body pose.",
    },
    "C": {
        "title": "C. Diffusion Model",
        "short": "Diffusion",
        "variants": ["diff_realisticvision15", "diff_realvisxl", "diff_sdxl"],
        "note": "Which inpainting model gives the best quality/runtime tradeoff?",
    },
    "D": {
        "title": "D. Face-DragGAN Tuning",
        "short": "Face-DragGAN",
        "variants": ["face_draggan_fast_pti", "face_draggan_high_inversion", "face_draggan_lower_support"],
        "note": "Among non-default Face-DragGAN settings, which tuning choice is most useful?",
    },
}

PRIMARY_METRICS = {
    "boundary_ring_mae": "lower",
    "edited_region_lpips": "lower",
    "background_mae": "lower",
}

METRIC_LABELS = {
    "boundary_ring_mae": "Edge error\nMAE lower",
    "edited_region_lpips": "Perceptual diff\nLPIPS lower",
    "background_mae": "Background error\nMAE lower",
}

VARIANT_DISPLAY_NAMES = {
    # A. Segmentation
    "seg_yolo_only": "YOLO segmentation only",
    "seg_sam_bbox_prior": "SAM using YOLO bbox prior",
    "seg_yolo_sam_consensus": "YOLO + SAM consensus mask",
    "seg_current_conservative": "Conservative refined mask pipeline",
    # B. Pose / geometry
    "pose_no_adjustment": "Box placement without pose warp",
    "pose_face_only": "Face-DragGAN face only",
    "pose_micro_piecewise": "Keypoint-guided micro piecewise warp",
    "pose_face_draggan_current": "Face-DragGAN face + body pose warp",
    # C. Diffusion
    "diff_realisticvision15": "RealisticVision v5.1 inpainting",
    "diff_realvisxl": "RealVisXL V4 inpainting",
    "diff_sdxl": "SDXL inpainting",
    # D. Post-process
    "harmonization_enabled": "LAB harmonization + light sharpening",
    "harmonization_disabled": "No post-generation harmonization",
    # E. Face-DragGAN tuning
    "face_draggan_current": "Face-DragGAN nose drag + PTI",
    "face_draggan_fast_pti": "Face-DragGAN fast PTI setting",
    "face_draggan_high_inversion": "Face-DragGAN high inversion steps",
    "face_draggan_lower_support": "Face-DragGAN lower support movement",
    # Legacy aliases that may still appear in old metrics rows.
    "current_main": "Main pipeline: conservative mask + micro warp + RealisticVision + harmonization",
    "no_harmonization": "No post-generation harmonization",
    "no_dog_edge_generation": "Generation mask excludes dog edge",
}

VARIANT_METHOD_DETAILS = {
    "seg_current_conservative": "YOLO/SAM candidate scoring, bbox prior, conservative mask refinement, and optional matting-edge alpha.",
    "pose_micro_piecewise": "Similarity alignment plus small keypoint-guided piecewise affine deformation with safety limits.",
    "pose_face_only": "AFHQ-dog StyleGAN inversion and nose-guided face drag, followed by bbox placement without body/leg pose warp.",
    "pose_face_draggan_current": "AFHQ-dog StyleGAN inversion, nose-guided face drag, then body/leg pose warp and diffusion refinement.",
    "diff_sdxl": "Official SDXL inpainting preset used as the strongest local diffusion option in this comparison.",
    "face_draggan_fast_pti": "Face-DragGAN branch with reduced PTI cost; best Face-DragGAN score/runtime tradeoff in this run.",
}

def display_name(name):
    return VARIANT_DISPLAY_NAMES.get(name, name.replace("_", " "))

print(f"Loaded {len(rows)} metric rows from {METRIC_CSV.relative_to(PROJECT_ROOT)}")
for row in rows[:3]:
    print(row["variant"], row["pair_id"], {k: round(row[k], 4) for k in NUMERIC_COLUMNS[:3]})

In [ ]:
def mean(values):
    vals = [v for v in values if isinstance(v, (int, float)) and math.isfinite(v)]
    return float(np.mean(vals)) if vals else float("nan")

def score_values(values, direction):
    vals = np.array(values, dtype=float)
    valid = np.isfinite(vals)
    out = np.full(vals.shape, np.nan, dtype=float)
    if not valid.any():
        return out.tolist()
    lo = np.nanmin(vals)
    hi = np.nanmax(vals)
    if abs(hi - lo) < 1e-12:
        out[valid] = 1.0
    elif direction == "lower":
        out[valid] = (hi - vals[valid]) / (hi - lo)
    else:
        out[valid] = (vals[valid] - lo) / (hi - lo)
    return out.tolist()

def group_rows(group_key):
    variants = set(GROUPS[group_key]["variants"])
    return [r.copy() for r in rows if r["variant"] in variants]

def score_group(group_key):
    cfg = GROUPS[group_key]
    data = group_rows(group_key)
    if not data:
        return [], [], {}, {}

    for pair_id in sorted({r["pair_id"] for r in data}):
        pair_rows = [r for r in data if r["pair_id"] == pair_id]
        for metric, direction in PRIMARY_METRICS.items():
            scores = score_values([r[metric] for r in pair_rows], direction)
            for r, score in zip(pair_rows, scores):
                r[f"{metric}_score"] = score
        for r in pair_rows:
            r["pair_score"] = mean([r.get(f"{m}_score", float("nan")) for m in PRIMARY_METRICS])

    summaries = []
    pair_table = {variant: {} for variant in cfg["variants"]}
    metric_table = {variant: {} for variant in cfg["variants"]}
    for variant in cfg["variants"]:
        variant_rows = [r for r in data if r["variant"] == variant]
        if not variant_rows:
            continue
        summaries.append({
            "variant": variant,
            "display": display_name(variant),
            "method_detail": VARIANT_METHOD_DETAILS.get(variant, ""),
            "mean_score": mean([r["pair_score"] for r in variant_rows]),
            "runtime_min": mean([r["elapsed_min"] for r in variant_rows]),
            "boundary_mae": mean([r["boundary_ring_mae"] for r in variant_rows]),
            "lpips": mean([r["edited_region_lpips"] for r in variant_rows]),
            "edited_ssim": mean([r["edited_region_ssim"] for r in variant_rows]),
            "background_mae": mean([r["background_mae"] for r in variant_rows]),
            "dog_ssim": mean([r["dog_region_ssim"] for r in variant_rows]),
            "n_pairs": len({r["pair_id"] for r in variant_rows}),
        })
        for pair in PAIR_ORDER:
            pair_table[variant][pair] = mean([r["pair_score"] for r in variant_rows if r["pair_id"] == pair])
        for metric in PRIMARY_METRICS:
            metric_table[variant][metric] = mean([r.get(f"{metric}_score", float("nan")) for r in variant_rows])

    summaries = sorted(summaries, key=lambda r: r["mean_score"], reverse=True)
    return data, summaries, pair_table, metric_table

GROUP_RESULTS = {key: score_group(key) for key in GROUPS}

best_summary = []
for key, (_, summaries, _, _) in GROUP_RESULTS.items():
    if summaries:
        best = summaries[0]
        best_summary.append({"group": GROUPS[key]["short"], **best})

print("Best variant per group:")
for row in best_summary:
    print(f"{row['group']:<14} {display_name(row['variant']):<52} score={row['mean_score']:.3f} runtime={row['runtime_min']:.1f} min")
    if row.get("method_detail"):
        print(f"{'':<14} detail: {row['method_detail']}")

## One-Slide Summary

This chart summarizes the **best-performing method in each technical component**.

What the chart shows:

- Each horizontal bar is the best variant within one ablation group.
- The x-axis is the normalized cycle score. Higher means better end-to-end cycle consistency.
- The text label on the right gives the concrete method name and its average runtime.

How to read it:

- A longer green bar means that component's best method performs strongly relative to its challengers.
- Runtime helps explain whether the win is practical. For example, a method can score well but still be too slow for routine use.
- This chart is best for a slide-level overview before showing the per-group details.

Important note:

- Scores are normalized **within each group**, so this plot is best for explaining each group's winner. It should not be read as a universal absolute quality score across unrelated tasks.


In [ ]:
def save_current_fig(name):
    path = FIGURE_DIR / name
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved: {path.relative_to(PROJECT_ROOT)}")

fig, ax = plt.subplots(figsize=(12, 5.8))
plot_rows = sorted(best_summary, key=lambda r: r["mean_score"])
colors = plt.cm.Greens(np.linspace(0.45, 0.88, len(plot_rows)))
bars = ax.barh([r["group"] for r in plot_rows], [r["mean_score"] for r in plot_rows], color=colors)
ax.set_xlim(0, 1.05)
ax.set_xlabel("Normalized cycle score, higher is better")
ax.set_title("Best Variant Per Technical Component")
ax.grid(axis="x", alpha=0.25)

for bar, row in zip(bars, plot_rows):
    label = f"{display_name(row['variant'])}  |  {row['runtime_min']:.1f} min"
    ax.text(bar.get_width() + 0.018, bar.get_y() + bar.get_height()/2, label, va="center", fontsize=10)

plt.tight_layout()
save_current_fig("01_best_variant_summary.png")
plt.show()

## Group-Level Figures

Each ablation group is now split into three separate figures instead of one crowded dashboard. This makes the labels larger and prevents long method names from overlapping the bars or heatmap.

Figure 1: `Cycle score`

- Shows the average normalized quality score across the evaluated pairs.
- Higher is better.
- The best method in the group is highlighted in dark green.
- This is the main winner/loser view for each technical comparison.

Figure 2: `Runtime`

- Shows average full-cycle runtime in minutes.
- Lower is faster.
- Runtime is intentionally separate from quality because speed and image quality are a tradeoff, not the same objective.

Figure 3: `Per-pair score`

- Shows whether each method is stable across `pair_1`, `pair_2`, and `face_pair`.
- Green means better score for that pair; red means worse.
- This is useful for spotting methods that only work on one easy case but fail on others.

Group-specific interpretation:

- `A. Segmentation Strategy`: compares how the dog mask is produced.
- `B. Pose / Geometry Adjustment`: compares three active geometry choices: Face-DragGAN face only, body pose only, and Face-DragGAN face + body pose.
- `C. Diffusion Model`: compares the inpainting model used for local repair.
- `D. Face-DragGAN Tuning`: compares non-default inversion and drag parameter choices in the face pipeline.

Default/no-op baselines are intentionally omitted from the winner charts. They are useful negative controls, but they should not be presented as candidate technical enhancements. The harmonization ablation is also omitted because its strongest result was `No post-generation harmonization`.

Diffusion model settings used in Group C:

| Variant | Model ID | Steps | Max generation side | Guidance scale | Strength |
| --- | --- | ---: | ---: | ---: | ---: |
| `RealisticVision v5.1 inpainting` | `Uminosachi/realisticVisionV51_v51VAE-inpainting` | 100 | 768 | 6.5 | 0.72 |
| `RealVisXL V4 inpainting` | `OzzyGT/RealVisXL_V4.0_inpainting` | 28 | 1024 | 6.5 | 0.72 |
| `SDXL inpainting` | `diffusers/stable-diffusion-xl-1.0-inpainting-0.1` | 28 | 1024 | 6.5 | 0.72 |

Shared diffusion mask/compositing settings:

- `diffusion_mask_mode = residual_plus_outer`: diffusion repairs residual blank/unstable regions plus an outer seam band.
- `generation_mask_include_dog_edge = True`: the generation mask may include a thin dog-edge band so the model can repair seam context.
- `generation_mask_outer_dilate_kernel = 65`: expands the generation area around the seam.
- `generation_mask_dog_edge_kernel = 13`: defines the thin dog-edge generation band.
- `dog_core_protect_kernel = 25`: protects the dog core from being rewritten.
- `commit_mask_outer_dilate_kernel = 11`: only this smaller commit region is blended back into the image.
- `commit_mask_blur = 21`: softens the commit boundary.
- Positive prompt: `photorealistic local background repair, seamless wall pavement grass, preserve the pasted dog unchanged`.
- Negative prompt: `extra dog, duplicate dog, dog fragments, extra limbs, extra head, changed dog body, wrong anatomy, blurry halo, gray outline, cartoon, text, watermark`.

The key design is that the model can generate using a larger context mask, but only the smaller commit mask is accepted back into the final image. This reduces hard seams while avoiding unnecessary changes to the dog body and surrounding background.


In [ ]:
def annotate_bars(ax, bars, fmt="{:.2f}", pad=0.012, fontsize=13):
    xmax = ax.get_xlim()[1]
    for bar in bars:
        width = bar.get_width()
        ax.text(
            width + xmax * pad,
            bar.get_y() + bar.get_height() / 2,
            fmt.format(width),
            va="center",
            fontsize=fontsize,
            fontweight="semibold",
        )

def matrix_from_table(table, row_order, col_order):
    return np.array([[table.get(row, {}).get(col, np.nan) for col in col_order] for row in row_order], dtype=float)

def draw_heatmap(ax, matrix, row_labels, col_labels, title):
    im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_title(title, fontsize=20, fontweight="bold", pad=16)
    ax.set_yticks(np.arange(len(row_labels)), row_labels, fontsize=13)
    ax.set_xticks(np.arange(len(col_labels)), col_labels, fontsize=14)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            if math.isfinite(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=14, fontweight="bold")
    return im

def short_method_label(variant):
    label = display_name(variant)
    replacements = {
        "Conservative refined mask pipeline": "Conservative refined mask",
        "Keypoint-guided micro piecewise warp": "Pose only",
        "Box placement without pose warp": "No pose warp",
        "Face-DragGAN face only": "Face only",
        "Face-DragGAN face + body pose warp": "Face + pose",
        "RealisticVision v5.1 inpainting": "RealisticVision v5.1",
        "RealVisXL V4 inpainting": "RealVisXL V4",
        "SDXL inpainting": "SDXL",
        "LAB harmonization + light sharpening": "LAB harmonization",
        "No post-generation harmonization": "No harmonization",
        "Face-DragGAN nose drag + PTI": "Face-DragGAN + PTI",
        "Face-DragGAN fast PTI setting": "Fast PTI",
        "Face-DragGAN high inversion steps": "High inversion",
        "Face-DragGAN lower support movement": "Lower support movement",
        "YOLO segmentation only": "YOLO only",
        "SAM using YOLO bbox prior": "SAM + bbox prior",
        "YOLO + SAM consensus mask": "YOLO + SAM consensus",
    }
    return replacements.get(label, label)

def wrapped_labels(variants, width=24):
    return ["\n".join(textwrap.wrap(short_method_label(v), width)) for v in variants]

def plot_group_composite_score(group_key):
    cfg = GROUPS[group_key]
    _, summaries, _, _ = GROUP_RESULTS[group_key]
    if not summaries:
        print(f"No rows for group {group_key}")
        return

    score_rows = sorted(summaries, key=lambda r: r["mean_score"])
    variants = [r["variant"] for r in score_rows]
    colors = ["#9ecae1"] * len(score_rows)
    colors[-1] = "#238b45"

    fig_height = max(5.8, 1.15 * len(score_rows) + 2.4)
    fig, ax = plt.subplots(figsize=(18, fig_height))
    bars = ax.barh(wrapped_labels(variants, 28), [r["mean_score"] for r in score_rows], color=colors, height=0.62)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("Normalized cycle score, higher is better", fontsize=15)
    ax.set_title(f"{cfg['title']} - Cycle Score", fontsize=22, fontweight="bold", pad=18)
    ax.grid(axis="x", alpha=0.25)
    ax.tick_params(axis="y", labelsize=14)
    ax.tick_params(axis="x", labelsize=13)
    annotate_bars(ax, bars, fmt="{:.2f}", fontsize=14)
    fig.text(0.01, 0.01, cfg["note"], fontsize=12, color="#444444")
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    save_current_fig(f"02_group_{group_key.lower()}_composite_score.png")
    plt.show()

def plot_group_runtime(group_key):
    cfg = GROUPS[group_key]
    _, summaries, _, _ = GROUP_RESULTS[group_key]
    if not summaries:
        print(f"No rows for group {group_key}")
        return

    time_rows = sorted(summaries, key=lambda r: r["runtime_min"], reverse=True)
    variants = [r["variant"] for r in time_rows]
    values = [r["runtime_min"] for r in time_rows]
    max_runtime = max([v for v in values if math.isfinite(v)] or [1.0])

    fig_height = max(5.8, 1.15 * len(time_rows) + 2.4)
    fig, ax = plt.subplots(figsize=(18, fig_height))
    bars = ax.barh(wrapped_labels(variants, 28), values, color="#fdae6b", height=0.62)
    ax.set_xlim(0, max_runtime * 1.18 if max_runtime > 0 else 1)
    ax.set_xlabel("Average full-cycle runtime, minutes. Lower is faster", fontsize=15)
    ax.set_title(f"{cfg['title']} - Runtime", fontsize=22, fontweight="bold", pad=18)
    ax.grid(axis="x", alpha=0.25)
    ax.tick_params(axis="y", labelsize=14)
    ax.tick_params(axis="x", labelsize=13)
    annotate_bars(ax, bars, fmt="{:.1f}", pad=0.01, fontsize=14)
    fig.text(0.01, 0.01, "Runtime is shown separately from quality so the tradeoff is visible rather than hidden inside the cycle score.", fontsize=12, color="#444444")
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    save_current_fig(f"02_group_{group_key.lower()}_runtime.png")
    plt.show()

def plot_group_per_pair_score(group_key):
    cfg = GROUPS[group_key]
    _, summaries, pair_table, _ = GROUP_RESULTS[group_key]
    if not summaries:
        print(f"No rows for group {group_key}")
        return

    heat_rows = [r["variant"] for r in summaries]
    heat_cols = [p for p in PAIR_ORDER if any(math.isfinite(pair_table.get(v, {}).get(p, np.nan)) for v in heat_rows)]
    heat = matrix_from_table(pair_table, heat_rows, heat_cols)

    fig_height = max(5.8, 0.98 * len(heat_rows) + 2.8)
    fig_width = max(12, 2.4 * len(heat_cols) + 9.5)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    im = draw_heatmap(ax, heat, wrapped_labels(heat_rows, 30), heat_cols, f"{cfg['title']} - Per-Pair Score")
    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.04)
    cbar.set_label("Normalized pair score, higher is better", fontsize=13)
    cbar.ax.tick_params(labelsize=12)
    fig.text(0.01, 0.01, "This view checks stability: a reliable method should stay green across pairs, not win only one example.", fontsize=12, color="#444444")
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    save_current_fig(f"02_group_{group_key.lower()}_per_pair_score.png")
    plt.show()

def plot_group_figures(group_key):
    plot_group_composite_score(group_key)
    plot_group_runtime(group_key)
    plot_group_per_pair_score(group_key)

for key in GROUPS:
    plot_group_figures(key)


## Metric Breakdown Heatmaps

These heatmaps explain **why** a method wins or loses.

Rows and columns:

- Each row is one variant inside the ablation group.
- Each column is one normalized metric.
- Green means that variant is strong for that metric.
- Red means that variant is weak for that metric.

Metric columns:

- `Edge error / MAE lower`: Boundary cleanliness. Good scores mean fewer halos, seams, and edge artifacts.
- `Perceptual diff / LPIPS lower`: Human-perceptual similarity. Good scores mean the edited region looks more visually similar after cycle reconstruction.
- `Background error / MAE lower`: Background preservation. Good scores mean the method does not unnecessarily damage non-dog pixels.

How to use this figure:

- If a method wins overall, this heatmap shows whether it wins because of boundary quality, perceptual similarity, or background preservation.
- If a method loses, this heatmap reveals the failure mode.
- This is useful for discussion because different CV methods often trade one failure mode for another.


In [ ]:
def plot_metric_breakdown(group_key):
    cfg = GROUPS[group_key]
    _, summaries, _, metric_table = GROUP_RESULTS[group_key]
    if not summaries:
        return

    row_order = [r["variant"] for r in summaries]
    col_order = list(PRIMARY_METRICS.keys())
    matrix = matrix_from_table(metric_table, row_order, col_order)

    fig, ax = plt.subplots(figsize=(12.5, max(3.5, 0.7 * len(row_order) + 2.0)))
    im = draw_heatmap(ax, matrix, [display_name(v) for v in row_order], [METRIC_LABELS[c] for c in col_order], f"{cfg['title']} - Metric Drivers")
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Normalized score")
    plt.tight_layout()
    save_current_fig(f"03_group_{group_key.lower()}_metric_breakdown.png")
    plt.show()

for key in GROUPS:
    plot_metric_breakdown(key)

## Quality vs Runtime Tradeoff

This scatter plot compares quality against compute cost.

Axes:

- x-axis: Average runtime per full cycle in minutes. Lower is better.
- y-axis: Normalized cycle quality score. Higher is better.

How to read it:

- Upper-left is ideal: high quality and low runtime.
- Upper-right means high quality but expensive.
- Lower-left means fast but lower quality.
- Lower-right is usually undesirable: slow and low quality.

Point labels:

- Each point is a concrete method variant.
- Point color indicates the ablation group: segmentation, pose, diffusion, or Face-DragGAN.

Diffusion-related interpretation:

- `RealisticVision v5.1 inpainting`: SD 1.5-style inpainting baseline; 100 steps, 768 max side, guidance 6.5, strength 0.72.
- `RealVisXL V4 inpainting`: realistic SDXL-family inpainting model; 28 steps, 1024 max side, guidance 6.5, strength 0.72.
- `SDXL inpainting`: official SDXL inpaint baseline; 28 steps, 1024 max side, guidance 6.5, strength 0.72.

Why this figure is important:

- It prevents us from choosing a method only because it has the best quality score.
- For a real pipeline, speed matters. A method that is only slightly better but 10x slower may not be practical.
- In this project, this chart is especially useful for comparing diffusion models and Face-DragGAN variants.


In [ ]:
trade_rows = []
for key, cfg in GROUPS.items():
    _, summaries, _, _ = GROUP_RESULTS[key]
    for row in summaries:
        trade_rows.append({**row, "group": cfg["short"]})

fig, ax = plt.subplots(figsize=(13, 7))
palette = dict(zip([GROUPS[k]["short"] for k in GROUPS], plt.cm.Set2(np.linspace(0, 1, len(GROUPS)))))
for group_name in sorted({r["group"] for r in trade_rows}):
    group_rows = [r for r in trade_rows if r["group"] == group_name]
    xs = [r["runtime_min"] for r in group_rows]
    ys = [r["mean_score"] for r in group_rows]
    ax.scatter(xs, ys, s=130, alpha=0.85, label=group_name, color=palette[group_name], edgecolor="white", linewidth=0.8)
    for row in group_rows:
        label = "\n".join(textwrap.wrap(display_name(row["variant"]), 16))
        ax.annotate(label, (row["runtime_min"], row["mean_score"]), xytext=(7, 5), textcoords="offset points", fontsize=8)

ax.set_xlabel("Average runtime per cycle, minutes")
ax.set_ylabel("Cycle score, higher is better")
ax.set_title("Quality vs Runtime Tradeoff")
ax.grid(alpha=0.25)
ax.legend(title="Comparison group", loc="lower right")
plt.tight_layout()
save_current_fig("04_quality_runtime_tradeoff.png")
plt.show()

## Exact Numbers For The Report

This table contains the exact values behind the charts.

Columns:

- `group`: Which ablation group the method belongs to.
- `variant`: Machine-readable experiment id used by the runner.
- `method`: Human-readable method name for slides and reports.
- `method_detail`: Extra explanation for methods that would otherwise sound ambiguous.
- `n_pairs`: Number of evaluated pairs.
- `mean_score`: Mean normalized cycle score. Higher is better.
- `runtime_min`: Average full-cycle runtime in minutes. Lower is faster.
- `boundary_mae`: Raw edge-region pixel error. Lower is better.
- `lpips`: Raw edited-region perceptual distance. Lower is better.
- `background_mae`: Raw nearby-background pixel error. Lower is better.

Diffusion configuration columns are not repeated in the table because all diffusion parameters are fixed in the Group C explanation above. The only changed variable in Group C is the inpainting model preset:

- `diff_realisticvision15`: `Uminosachi/realisticVisionV51_v51VAE-inpainting`, 100 steps, max side 768, guidance 6.5, strength 0.72.
- `diff_realvisxl`: `OzzyGT/RealVisXL_V4.0_inpainting`, 28 steps, max side 1024, guidance 6.5, strength 0.72.
- `diff_sdxl`: `diffusers/stable-diffusion-xl-1.0-inpainting-0.1`, 28 steps, max side 1024, guidance 6.5, strength 0.72.

Use this table when you need exact numbers in a written report. Use the charts when you need fast visual communication.


In [ ]:
final_rows = []
for key, cfg in GROUPS.items():
    _, summaries, _, _ = GROUP_RESULTS[key]
    for row in summaries:
        final_rows.append({
            "group": cfg["short"],
            "variant": row["variant"],
            "method": display_name(row["variant"]),
            "method_detail": row.get("method_detail", ""),
            "n_pairs": row["n_pairs"],
            "mean_score": row["mean_score"],
            "runtime_min": row["runtime_min"],
            "boundary_mae": row["boundary_mae"],
            "lpips": row["lpips"],
            "background_mae": row["background_mae"],
        })

out_csv = FIGURE_DIR / "comparison_visual_summary.csv"
with open(out_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()))
    writer.writeheader()
    writer.writerows(final_rows)

print(f"Saved table: {out_csv.relative_to(PROJECT_ROOT)}")
print("\nGroup | Method | Score | Runtime | Boundary MAE | LPIPS | Background MAE")
print("-" * 122)
for row in final_rows:
    print(
        f"{row['group']:<14} | {row['method']:<52} | {row['mean_score']:.3f} | {row['runtime_min']:>6.2f} | "
        f"{row['boundary_mae']:>8.2f} | {row['lpips']:.4f} | {row['background_mae']:>8.2f}"
    )

## Suggested Presentation Takeaways

- Use the one-slide summary to introduce the winning method in each technical component.
- Use group dashboards to explain tradeoffs inside each ablation.
- Use metric breakdown heatmaps to explain why a method wins or loses.
- Use the quality/runtime scatterplot to justify whether expensive methods are worth keeping.

All generated figures are saved under `data/outputs/comparison_eval/figures/`.